In [120]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

SHOW_GRAPHS = True

# Preparation of Data Frames

In [121]:
df  = pd.DataFrame()

## Sentiment

In [122]:
# Reading the saved data pickle file

df_sentiment  = pd.DataFrame()
df_time  = pd.DataFrame()
df_hour  = pd.DataFrame()

df_sentiment=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentHourly.pkl', compression=None)
# df_sentiment["sentiment"] = df_sentiment["pos"]*100
# df_sentiment.drop(columns=["compound", "neg", "pos", "neu", "Articles", "Title", "Text", "Company"], inplace=True)
# df_sentiment.index = pd.to_datetime(df_sentiment.Date)
df_sentiment.Date = pd.to_datetime(df_sentiment.Date)
# df_sentiment["sentiment"].ffill(inplace=True)

# df_sentiment["sentiment_yesterday"] = df_sentiment["sentiment"].shift(periods=1)
# df_sentiment["sentiment_day_before_yesterday"] = df_sentiment["sentiment"].shift(periods=2)

# # SENTIMENT RETURN
# df_sentiment["sentiment_return"]=(df_sentiment["sentiment_yesterday"]-df_sentiment["sentiment_day_before_yesterday"])/df_sentiment["sentiment_day_before_yesterday"]
# df_sentiment["sentiment_return"].ffill(inplace=True)

# # SENTIMENT LN RETURN
# df_sentiment["sentiment_ln_return"]=np.log(df_sentiment["sentiment_yesterday"]/df_sentiment["sentiment_day_before_yesterday"])
# df_sentiment["sentiment_ln_return"].ffill(inplace=True)

# # SENTIMENT EWM
# df_sentiment["sentiment_ewm"]=df_sentiment["sentiment"].ewm(10).mean()
# df_sentiment["sentiment_ewm"].ffill(inplace=True)

# # SENTIMENT EWM YESTERDAY
# df_sentiment["sentiment_yesterday_ewm"]=df_sentiment["sentiment_yesterday"].ewm(10).mean()
# df_sentiment["sentiment_yesterday_ewm"].ffill(inplace=True)


In [123]:
df_sentiment["sentiment(t-1)"] = df_sentiment["sentiment"].shift(periods=1)
df_sentiment["sentiment(t-2)"] = df_sentiment["sentiment"].shift(periods=2)
df_sentiment["sentiment(t-1)"] = df_sentiment["sentiment(t-1)"].ffill()
df_sentiment["sentiment(t-2)"] = df_sentiment["sentiment(t-2)"].ffill()

# SENTIMENT RETURN
df_sentiment["sentiment_return"]=(df_sentiment["sentiment(t-1)"]-df_sentiment["sentiment(t-2)"])/df_sentiment["sentiment(t-2)"]
# SENTIMENT LN RETURN
df_sentiment["sentiment_ln_return"]=np.log(df_sentiment["sentiment(t-1)"]/df_sentiment["sentiment(t-2)"])
# SENTIMENT EWM
df_sentiment["sentiment_ewm10"]=df_sentiment["sentiment"].ewm(10).mean()
df_sentiment["sentiment_ewm20"]=df_sentiment["sentiment"].ewm(20).mean()

# POSITIVE LN RETURN
df_sentiment["pos(t-1)"] = df_sentiment["pos"].shift(periods=1)
df_sentiment["pos(t-2)"] = df_sentiment["pos"].shift(periods=2)
df_sentiment["pos(t-1)"] = df_sentiment["pos(t-1)"].ffill()
df_sentiment["pos(t-2)"] = df_sentiment["pos(t-2)"].ffill()
df_sentiment["pos_ln_return"]=np.log(df_sentiment["pos(t-1)"]/df_sentiment["pos(t-2)"])

# NEGATIVE LN RETURN
df_sentiment["neg(t-1)"] = df_sentiment["neg"].shift(periods=1)
df_sentiment["neg(t-2)"] = df_sentiment["neg"].shift(periods=2)
df_sentiment["neg(t-1)"] = df_sentiment["neg(t-1)"].ffill()
df_sentiment["neg(t-2)"] = df_sentiment["neg(t-2)"].ffill()
df_sentiment["neg_ln_return"]=np.log(df_sentiment["neg(t-1)"]/df_sentiment["neg(t-2)"])

# NEUTRAL LN RETURN
df_sentiment["neu(t-1)"] = df_sentiment["neu"].shift(periods=1)
df_sentiment["neu(t-2)"] = df_sentiment["neu"].shift(periods=2)
df_sentiment["neu(t-1)"] = df_sentiment["neu(t-1)"].ffill()
df_sentiment["neu(t-2)"] = df_sentiment["neu(t-2)"].ffill()
df_sentiment["neu_ln_return"]=np.log(df_sentiment["neu(t-1)"]/df_sentiment["neu(t-2)"])

# COMPOUND LN RETURN
df_sentiment["compound(t-1)"] = df_sentiment["compound"].shift(periods=1)
df_sentiment["compound(t-2)"] = df_sentiment["compound"].shift(periods=2)
df_sentiment["compound(t-1)"] = df_sentiment["compound(t-1)"].ffill()
df_sentiment["compound(t-2)"] = df_sentiment["compound(t-2)"].ffill()
df_sentiment["compound_ln_return"]=np.log(df_sentiment["compound(t-1)"]/df_sentiment["compound(t-2)"])

# PRESS RELEASE COUNT
df_sentiment["PressReleaseCount(t-1)"] = df_sentiment["PressReleaseCount"].shift(periods=1)
df_sentiment["PressReleaseCount(t-2)"] = df_sentiment["PressReleaseCount"].shift(periods=2)
df_sentiment["PressReleaseCount(t-1)"] = df_sentiment["PressReleaseCount(t-1)"].ffill()
df_sentiment["PressReleaseCount(t-2)"] = df_sentiment["PressReleaseCount(t-2)"].ffill()
df_sentiment["PressReleaseCount_ln_return"]=np.log(df_sentiment["PressReleaseCount(t-1)"]/df_sentiment["PressReleaseCount(t-2)"])
df_sentiment = df_sentiment.dropna()
# df_sentiment = df_sentiment.ffill()

In [124]:
df_sentiment.loc[df_sentiment["Date"]=="2020-06-01"]

,Date,Hour,compound,neg,neu,pos,sentiment,polarity,subjectivity,polarityB,...,neg_ln_return,neu(t-1),neu(t-2),neu_ln_return,compound(t-1),compound(t-2),compound_ln_return,PressReleaseCount(t-1),PressReleaseCount(t-2),PressReleaseCount_ln_return
4060,2020-06-01,12,0.0000,0.000,0.000,0.000,0.000000,0.000000,0.000000,Neutral,...,inf,0.848,0.930,-0.092304,0.9563,0.7003,0.311563,2.0,3.0,-0.405465
4061,2020-06-01,13,0.9813,0.008,0.878,0.115,93.495935,0.050810,0.364170,Positive,...,-inf,0.000,0.848,-inf,0.0000,0.9563,-inf,0.0,2.0,-inf
4062,2020-06-01,14,0.0000,0.000,0.000,0.000,0.000000,0.000000,0.000000,Neutral,...,inf,0.878,0.000,inf,0.9813,0.0000,inf,2.0,0.0,inf
4063,2020-06-01,15,0.0000,0.000,0.000,0.000,0.000000,0.000000,0.000000,Neutral,...,-inf,0.000,0.878,-inf,0.0000,0.9813,-inf,0.0,2.0,-inf
4070,2020-06-01,21,0.8625,0.000,0.851,0.149,100.000000,-0.035714,0.294643,Negative,...,inf,0.794,0.000,inf,0.9940,0.0000,inf,3.0,0.0,inf
4071,2020-06-01,22,0.0000,0.000,0.000,0.000,0.000000,0.000000,0.000000,Neutral,...,-inf,0.851,0.794,0.069329,0.8625,0.9940,-0.141902,1.0,3.0,-1.098612
4075,2020-06-01,5,0.0000,0.000,0.000,0.000,0.000000,0.000000,0.000000,Neutral,...,inf,0.793,0.000,inf,0.8074,0.0000,inf,2.0,0.0,inf
4076,2020-06-01,6,0.0000,0.000,0.000,0.000,0.000000,0.000000,0.000000,Neutral,...,-inf,0.000,0.793,-inf,0.0000,0.8074,-inf,0.0,2.0,-inf
4079,2020-06-01,9,0.8020,0.000,0.942,0.058,100.000000,-0.007576,0.405303,Negative,...,inf,0.924,0.000,inf,0.5574,0.0000,inf,1.0,0.0,inf


In [125]:
df = df_sentiment

## VIX

In [126]:
# from yahoo_fin.stock_info import get_data

# df_vix = pd.DataFrame(data=get_data("^VIX", start_date="25/09/2019", end_date="09/29/2021", index_as_date = True, interval="1d"))
# df_vix.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/vix.pkl")

In [127]:
df_vix=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/vix.pkl', compression=None)

df_vix.index = pd.to_datetime(df_vix.index)
df_vix["Date"]= pd.to_datetime(df_vix.index)

df_vix["vix"] = df_vix["adjclose"]
df_vix.drop(columns=["open", "high", "low", "close", "adjclose", "ticker", "volume"], inplace=True)
df_vix["vix"]  = df_vix["vix"].ffill()

df_vix["vix_yesterday"] = df_vix["vix"].shift(periods=1)
df_vix["vix_yesterday"].ffill(inplace=True)

# vix RETURN
df_vix["vix_return"]=(df_vix["vix"]-df_vix["vix_yesterday"])/df_vix["vix_yesterday"]
df_vix["vix_return"].ffill(inplace=True)

# vix LN RETURN
df_vix["vix_ln_return"]=np.log(df_vix["vix"]/df_vix["vix_yesterday"])
df_vix["vix_ln_return"].ffill(inplace=True)

# vix EWM
df_vix["vix_ewm"]=df_vix["vix"].ewm(10).mean()
df_vix["vix_ewm"].ffill(inplace=True)


In [128]:
df_vix

,Date,vix,vix_yesterday,vix_return,vix_ln_return,vix_ewm
2019-09-25,2019-09-25,15.960000,NaN,NaN,NaN,15.960000
2019-09-26,2019-09-26,16.070000,15.960000,0.006892,0.006869,16.017619
2019-09-27,2019-09-27,17.219999,16.070000,0.071562,0.069117,16.457160
2019-09-30,2019-09-30,16.240000,17.219999,-0.056911,-0.058594,16.394880
2019-10-01,2019-10-01,18.559999,16.240000,0.142857,0.133531,16.914110
...,...,...,...,...,...,...
2021-09-22,2021-09-22,20.870001,24.360001,-0.143268,-0.154630,19.646968
2021-09-23,2021-09-23,18.629999,20.870001,-0.107331,-0.113540,19.554516
2021-09-24,2021-09-24,17.750000,18.629999,-0.047236,-0.048388,19.390469
2021-09-27,2021-09-27,18.760000,17.750000,0.056901,0.055341,19.333154


In [129]:
df.Date

16      2019-09-25
17      2019-09-25
24      2019-09-26
25      2019-09-26
29      2019-09-26
           ...    
11645   2021-09-28
11646   2021-09-28
11659   2021-09-28
11662   2021-09-28
11663   2021-09-28
Name: Date, Length: 5476, dtype: datetime64[ns]

In [130]:
df

,Date,Hour,compound,neg,neu,pos,sentiment,polarity,subjectivity,polarityB,...,neg_ln_return,neu(t-1),neu(t-2),neu_ln_return,compound(t-1),compound(t-2),compound_ln_return,PressReleaseCount(t-1),PressReleaseCount(t-2),PressReleaseCount_ln_return
16,2019-09-25,23,0.0000,0.000,0.000,0.000,0.000000,0.000000,0.000000,Neutral,...,inf,0.875,0.000,inf,1.0000,0.0000,inf,90.0,0.0,inf
17,2019-09-25,3,0.0000,0.000,0.000,0.000,0.000000,0.000000,0.000000,Neutral,...,-inf,0.000,0.875,-inf,0.0000,1.0000,-inf,0.0,90.0,-inf
24,2019-09-26,0,0.0000,0.000,0.000,0.000,0.000000,0.000000,0.000000,Neutral,...,inf,0.857,0.867,-0.011601,0.9828,0.9432,0.041127,4.0,2.0,0.693147
25,2019-09-26,1,0.0000,0.000,0.000,0.000,0.000000,0.000000,0.000000,Neutral,...,-inf,0.000,0.857,-inf,0.0000,0.9828,-inf,0.0,4.0,-inf
29,2019-09-26,13,0.0000,0.000,0.000,0.000,0.000000,0.000000,0.000000,Neutral,...,-inf,0.000,0.852,-inf,0.0000,0.9866,-inf,0.0,4.0,-inf
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11645,2021-09-28,13,0.9670,0.000,0.842,0.158,100.000000,0.138430,0.440220,Positive,...,inf,0.739,1.000,-0.302457,0.9810,0.0000,inf,3.0,1.0,1.098612
11646,2021-09-28,14,0.0000,0.000,0.000,0.000,0.000000,0.000000,0.000000,Neutral,...,-inf,0.842,0.739,0.130482,0.9670,0.9810,-0.014374,2.0,3.0,-0.405465
11659,2021-09-28,5,-0.7783,0.109,0.854,0.037,25.342466,0.048011,0.417330,Positive,...,inf,0.785,0.000,inf,0.9371,0.0000,inf,1.0,0.0,inf
11662,2021-09-28,8,0.9990,0.002,0.844,0.153,98.709677,0.106453,0.341056,Positive,...,-1.178655,0.841,0.872,-0.036198,0.9926,0.9892,0.003431,6.0,10.0,-0.510826


In [131]:
df_vix

,Date,vix,vix_yesterday,vix_return,vix_ln_return,vix_ewm
2019-09-25,2019-09-25,15.960000,NaN,NaN,NaN,15.960000
2019-09-26,2019-09-26,16.070000,15.960000,0.006892,0.006869,16.017619
2019-09-27,2019-09-27,17.219999,16.070000,0.071562,0.069117,16.457160
2019-09-30,2019-09-30,16.240000,17.219999,-0.056911,-0.058594,16.394880
2019-10-01,2019-10-01,18.559999,16.240000,0.142857,0.133531,16.914110
...,...,...,...,...,...,...
2021-09-22,2021-09-22,20.870001,24.360001,-0.143268,-0.154630,19.646968
2021-09-23,2021-09-23,18.629999,20.870001,-0.107331,-0.113540,19.554516
2021-09-24,2021-09-24,17.750000,18.629999,-0.047236,-0.048388,19.390469
2021-09-27,2021-09-27,18.760000,17.750000,0.056901,0.055341,19.333154


In [132]:
df = df_vix.merge(df,how='left',right_on="Date", left_on="Date")

## S&P500

In [133]:
# from yahoo_fin.stock_info import get_data

# df_sp500 = get_data("^GSPC", start_date="25/09/2019", end_date="09/29/2021", index_as_date = True, interval="1d")
# df_sp500.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sp500.pkl")

In [134]:
df_sp500=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sp500.pkl', compression=None)


df_sp500.index = pd.to_datetime(df_sp500.index)
df_sp500["Date"]= pd.to_datetime(df_sp500.index)

df_sp500["sp500"] = df_sp500["adjclose"]
df_sp500.drop(columns=["open", "high", "low", "close", "adjclose", "ticker", "volume"], inplace=True)
df_sp500["sp500"]  = df_sp500["sp500"].ffill()

df_sp500["sp500_yesterday"] = df_sp500["sp500"].shift(periods=1)
# sp500 RETURN
df_sp500["sp500_return"]=(df_sp500["sp500"]-df_sp500["sp500_yesterday"])/df_sp500["sp500_yesterday"]
# sp500 LN RETURN
df_sp500["sp500_ln_return"]=np.log(df_sp500["sp500"]/df_sp500["sp500_yesterday"])
# sp500 EWM
df_sp500["sp500_ewm"]=df_sp500["sp500"].ewm(10).mean()

In [135]:
df_sp500

,Date,sp500,sp500_yesterday,sp500_return,sp500_ln_return,sp500_ewm
2019-09-25,2019-09-25,2984.870117,NaN,NaN,NaN,2984.870117
2019-09-26,2019-09-26,2977.620117,2984.870117,-0.002429,-0.002432,2981.072498
2019-09-27,2019-09-27,2961.790039,2977.620117,-0.005316,-0.005331,2974.023623
2019-09-30,2019-09-30,2976.739990,2961.790039,0.005048,0.005035,2974.802655
2019-10-01,2019-10-01,2940.250000,2976.739990,-0.012258,-0.012334,2966.516379
...,...,...,...,...,...,...
2021-09-22,2021-09-22,4395.640137,4354.189941,0.009520,0.009475,4447.769295
2021-09-23,2021-09-23,4448.979980,4395.640137,0.012135,0.012062,4447.879358
2021-09-24,2021-09-24,4455.479980,4448.979980,0.001461,0.001460,4448.570323
2021-09-27,2021-09-27,4443.109863,4455.479980,-0.002776,-0.002780,4448.073918


In [136]:
df = pd.merge(df_sp500,df,how='outer', on='Date')

In [137]:
# df = df.drop([475])

# DAY MONTH YEAR

In [138]:
import datetime as dt
from datetime import date
import calendar
my_date = date.today()
calendar.day_name[my_date.weekday()]  #'Wednesday'

# Prepare data
df['year'] = [d.year for d in df.Date]
df['month'] = [d.strftime('%b') for d in df.Date]
df['day'] = [d.day_name() for d in df.Date]
years = df['year'].unique()
df['month-year'] = [d.strftime('%b') + str(d.year) for d in df.Date]

# MERGE

Missing Values, how to handle

Option 1: Fill NaN with Outlier or Zero
In this specific example filling the missing value with an outlier value such as np.inf or 0 seems to be very naive. However, using values like -999, is sometimes a good idea.

Option 2: Fill NaN with Mean Value
Filling NaNs with the mean value is also not sufficient and naive, and doesn't seems to be a good option.

Option 3: Fill NaN with Last Value with .ffill()
Filling NaNs with the last value could be bit better.

Option 4: Fill NaN with Linearly ffilld Value with .ffill()
Filling NaNs with the ffilld values is the best option in this small examlple but it requires knowledge of the neighouring value

# THREE PERIODS

In [139]:
df.index = df.Date

In [140]:
df_part1 = df.loc['2019-09-25':'2020-02-14']
df_part2 = df.loc['2020-02-15':'2020-05-31']
df_part3 = df.loc['2020-06-01':'2021-09-28']

In [141]:
# df_part3 = df_part3.ffill()

In [142]:
# df.reset_index(drop=True, inplace=True)
# df_part1.reset_index(drop=True, inplace=True)
# df_part2.reset_index(drop=True, inplace=True)
# df_part3.reset_index(drop=True, inplace=True)

In [143]:
# df.sort_index()

In [144]:
# df_part3['Num_word'] = df_part3['Articles'].apply(lambda x:len(str(x).split())) #Number Of words in main text

In [145]:
df_part3 = df_part3[df_part3["PressReleaseCount"]>0]


In [150]:
dfHour = pd.DataFrame()
dfHour.index = df_part3.loc[df_part3["Hour"]==0].Date

for i in (range(24)):

    dfHour["Hour"+str(i)+"_sentiment_ln_return"]=df_part3.loc[df_part3["Hour"]==i].sentiment_ln_return
    dfHour["Hour"+str(i)+"_sentiment_ln_return"].ffill(inplace=True)
    dfHour = dfHour[dfHour["Hour"+str(i)+"_sentiment_ln_return"]!=-np.inf]
    dfHour = dfHour[dfHour["Hour"+str(i)+"_sentiment_ln_return"]!=np.inf]
# maxnum = max(dfHour["Hour"+str(i)+"_sentiment"])
# minnum =min(dfHour["Hour"+str(i)+"_sentiment"])

    dfHour["Hour"+str(i)+"_sentiment_ln_return"] = dfHour["Hour"+str(i)+"_sentiment_ln_return"].replace(np.inf,0)

    dfHour["Hour"+str(i)+"_sentiment_ln_return"] = dfHour["Hour"+str(i)+"_sentiment_ln_return"].replace(-np.inf,0)
    # dfHour["Hour"+str(i)+"_vix_ln_return"]=df_part3.loc[df_part3["Hour"]==i].vix_ln_return
    # dfHour["Hour"+str(i)+"_vix_ln_return"].ffill(inplace=True)
    # mn = df_part3.loc[df_part3['pos_ln_return'] != -np.inf, 'pos_ln_return'].min()


# df_part3['sentiment_ln_return'].dropna()
# mx = df_part3.loc[df_part3['sentiment_ln_return'] != np.inf, 'sentiment_ln_return'].max()
# df_part3['sentiment_ln_return'].replace(np.inf,mx,inplace=True)
# mn = df_part3.loc[df_part3['sentiment_ln_return'] != -np.inf, 'sentiment_ln_return'].min()
# df_part3['sentiment_ln_return'].replace(-np.inf,mn,inplace=True)

# df_part3.ffill(inplace=True)


In [152]:
# dfHour.ist(figsize=[30,30],bins=120)

In [153]:
mx = df_part3.loc[df_part3['pos_ln_return'] != np.inf, 'pos_ln_return'].max()
df_part3['pos_ln_return'].replace(np.inf,0,inplace=True)
mn = df_part3.loc[df_part3['pos_ln_return'] != -np.inf, 'pos_ln_return'].min()
df_part3['pos_ln_return'].replace(-np.inf,0,inplace=True)

mx = df_part3.loc[df_part3['neg_ln_return'] != np.inf, 'pos_ln_return'].max()
df_part3['neg_ln_return'].replace(np.inf,0,inplace=True)
mn = df_part3.loc[df_part3['neg_ln_return'] != -np.inf, 'pos_ln_return'].min()
df_part3['neg_ln_return'].replace(-np.inf,0,inplace=True)

mx = df_part3.loc[df_part3['neu_ln_return'] != np.inf, 'neu_ln_return'].max()
df_part3['neu_ln_return'].replace(np.inf,0,inplace=True)
mn = df_part3.loc[df_part3['neu_ln_return'] != -np.inf, 'neu_ln_return'].min()
df_part3['neu_ln_return'].replace(-np.inf,0,inplace=True)

mx = df_part3.loc[df_part3['compound_ln_return'] != np.inf, 'compound_ln_return'].max()
df_part3['compound_ln_return'].replace(np.inf,0,inplace=True)
mn = df_part3.loc[df_part3['compound_ln_return'] != -np.inf, 'compound_ln_return'].min()
df_part3['compound_ln_return'].replace(-np.inf,0,inplace=True)

In [154]:
# mx = df_part3.loc[df_part3['sentiment'] != np.inf, 'pos_ln_return'].max()
# df_part3['neg_ln_return'].replace(np.inf,0,inplace=True)
# mn = df_part3.loc[df_part3['neg_ln_return'] != -np.inf, 'pos_ln_return'].min()
# df_part3['neg_ln_return'].replace(-np.inf,0,inplace=True)
df_part3 = df_part3[df_part3["PressReleaseCount"]>0]
df_part3['sentiment'].ffill()

Date
2020-06-01     93.495935
2020-06-01    100.000000
2020-06-01    100.000000
2020-06-02     84.939759
2020-06-02    100.000000
                 ...    
2021-09-28     95.785441
2021-09-28    100.000000
2021-09-28     25.342466
2021-09-28     98.709677
2021-09-28     83.766234
Name: sentiment, Length: 2738, dtype: float64